## Preenfasis

### Configuración

In [ ]:
from pathlib import Path

# Ruta base donde están TUS audios por género.
# Ejemplo con tu estructura: MusicaAndina/Seg_20s/Huaylash/...
# CAMBIA esta ruta a la que corresponda en tu máquina:
base_dir = Path(r'C:\Users\RICHI\Documents\Tesis\Data\Andinas\Segmentacion20seg')

# Extensiones a procesar
EXTS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a', '.aac'}

# Carpeta de salida (se crea en paralelo, preservando subcarpetas)
output_root = base_dir.parent / (base_dir.name + '_preenfasis')

# Parámetros de procesamiento
alpha = 0.97          # 0.95 -- 0.97 típico; a mayor alpha, más agudos
target_rms = None     # p.ej. 0.1 para fijar RMS; usa None para omitir
write_outputs = True  # True = guarda archivos en disco

base_dir, output_root

In [ ]:
# ====================== Utilidades ======================
import numpy as np
import librosa
import soundfile as sf

def peak_normalize(x: np.ndarray, eps: float = 1e-9) -> np.ndarray:
    """Escala la señal a rango [-1, 1] por pico absoluto."""
    peak = np.max(np.abs(x)) + eps
    return (x / peak).astype(np.float32)

def rms(x: np.ndarray, eps: float = 1e-12) -> float:
    return float(np.sqrt(np.mean(np.square(x)) + eps))

def rms_normalize(x: np.ndarray, target: float) -> np.ndarray:
    """Ajusta el RMS de 'x' a 'target'."""
    current = rms(x)
    if current == 0:
        return x.astype(np.float32)
    return (x * (target / current)).astype(np.float32)

def normalize_wave(x: np.ndarray, target_rms: float | None) -> np.ndarray:
    """
    Normalización combinada: 
      1) Peak a [-1, 1] (garantiza rango)
      2) (Opcional) RMS objetivo
      3) Clip de seguridad a [-1, 1]
    """
    y = peak_normalize(x)
    if target_rms is not None:
        y = rms_normalize(y, target_rms)
        y = np.clip(y, -1.0, 1.0)
    return y.astype(np.float32)

def pre_emphasis(x: np.ndarray, alpha: float = 0.97) -> np.ndarray:
    """Filtro de 1er orden: y[n] = x[n] - alpha * x[n-1]."""
    if x.ndim > 1:
        x = np.mean(x, axis=0)       # a mono si entra multicanal
    y = np.empty_like(x, dtype=np.float32)
    y[0] = x[0]
    y[1:] = x[1:] - alpha * x[:-1]
    return y

def load_audio(path: Path, sr: int | None = None) -> tuple[np.ndarray, int]:
    """Carga audio (mono) con su frecuencia de muestreo original si sr=None."""
    x, sr = librosa.load(str(path), sr=sr, mono=True)
    return x.astype(np.float32), sr

def save_audio(path: Path, x: np.ndarray, sr: int):
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), x.astype(np.float32), sr)  # WAV float32


In [ ]:
# ====================== Procesamiento en lote ======================
from tqdm.auto import tqdm

def process_dataset(input_root: Path, output_root: Path, alpha: float = 0.97, 
                    target_rms: float | None = None, write: bool = True):
    input_root = Path(input_root)
    output_root = Path(output_root)

    files = [p for p in input_root.rglob('*') if p.suffix.lower() in EXTS]
    if not files:
        print('No se encontraron archivos de audio en:', input_root)
        return

    for src in tqdm(files, desc='Procesando audios'):
        rel = src.relative_to(input_root)
        dst = output_root / rel.with_suffix('.wav')  # salida en WAV

        try:
            # 1) Carga
            x, sr = load_audio(src)
            # 2) Normalización (RMS opcional) a [-1,1]
            x_norm = normalize_wave(x, target_rms=target_rms)
            # 3) Pre-énfasis
            x_pre = pre_emphasis(x_norm, alpha=alpha)
            # 4) Re-normaliza pico para evitar clipping tras el filtro
            x_out = peak_normalize(x_pre)

            if write:
                save_audio(dst, x_out, sr)
        except Exception as e:
            print(f'Error en {src}: {e}')

# Ejecuta el pipeline completo
process_dataset(base_dir, output_root, alpha=alpha, target_rms=target_rms, write=write_outputs)
print('Salida en:', output_root)


In [ ]:
# ====================== Vista previa (opcional) ======================
import random
import matplotlib.pyplot as plt
from IPython.display import Audio, display

candidates = [p for p in base_dir.rglob('*') if p.suffix.lower() in EXTS]
if candidates:
    sample_path = random.choice(candidates)
    x, sr = load_audio(sample_path)
    x_n = normalize_wave(x, target_rms=target_rms)
    x_p = pre_emphasis(x_n, alpha=alpha)
    x_p = peak_normalize(x_p)  # para escuchar sin clipping

    # gráfico de los primeros 2 s
    n = min(len(x), sr * 2)
    t = np.arange(n) / sr
    plt.figure()
    plt.plot(t, x[:n], label='Original')
    plt.plot(t, x_p[:n], label='Con pre-énfasis')
    plt.title('Señal (primeros 2 s)')
    plt.xlabel('Tiempo [s]')
    plt.ylabel('Amplitud')
    plt.legend()
    plt.show()

    print('Archivo de muestra:', sample_path.name)
    print('Escuchar original:')
    display(Audio(x, rate=sr))
    print('Escuchar con pre-énfasis:')
    display(Audio(x_p, rate=sr))
else:
    print('No hay archivos para vista previa.')

## Split de datos

In [ ]:

from pathlib import Path
from collections import defaultdict
import shutil
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit

# ===== Config =====
input_root  = Path(r'C:\Users\RICHI\Documents\Tesis\Data\Andinas\Segmentacion20seg_preenfasis')   # carpeta con subcarpetas por género
output_root = input_root.parent / (input_root.name + '_split')
EXTS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a', '.aac'}

ratios = (0.70, 0.20, 0.10)   # train, val, test
seed = 42
copy_files = True

# ===== Listado (X=files, y=labels) =====
X, y = [], []
genres = sorted([d for d in input_root.iterdir() if d.is_dir()])
class_to_idx = {g.name: i for i, g in enumerate(genres)}

for g in genres:
    files = [p for p in g.rglob('*') if p.suffix.lower() in EXTS]
    X.extend(files)
    y.extend([class_to_idx[g.name]] * len(files))

X = np.array(X, dtype=object)
y = np.array(y)

# ===== Split estratificado 70/20/10 =====
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=ratios[2], random_state=seed)  # separar TEST (10%)
idx_trainval, idx_test = next(sss1.split(X, y))
X_trainval, y_trainval = X[idx_trainval], y[idx_trainval]
X_test, y_test = X[idx_test], y[idx_test]

val_ratio_rel = ratios[1] / (ratios[0] + ratios[1])  # 20/(70+20) = 0.222...
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio_rel, random_state=seed)
idx_train, idx_val = next(sss2.split(X_trainval, y_trainval))
X_train, y_train = X_trainval[idx_train], y_trainval[idx_train]
X_val, y_val     = X_trainval[idx_val],   y_trainval[idx_val]

# ===== Guardado a carpetas =====
def dump_split(split_name, files):
    for src in files:
        genre = src.relative_to(input_root).parts[0]     # primer nivel = género
        rel   = src.relative_to(input_root/genre)
        dst   = output_root / split_name / genre / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if copy_files:
            shutil.copy2(src, dst)

dump_split('train', X_train)
dump_split('val',   X_val)
dump_split('test',  X_test)

print('Hecho. Salida en:', output_root)


## Data Augmentation

### Configuración

In [ ]:
# %pip install librosa soundfile tqdm --quiet

from pathlib import Path

# Carpeta de TRAIN (la que salió del split)
train_root = Path(r'C:\Users\RICHI\Documents\Tesis\Data\Andinas\Segmentacion20seg_preenfasis_split\train')

# Carpeta de salida para los aumentos
out_root = train_root.parent / 'train_aug'

# Extensiones válidas
EXTS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a', '.aac'}

# Reproducibilidad y control
WRITE = True
SEED  = 123

### Utilidades y transformaciones

In [ ]:
import numpy as np
import librosa, soundfile as sf, random
from tqdm.auto import tqdm

random.seed(SEED)
rng = np.random.default_rng(SEED)

def load_audio(path, sr=None):
    x, sr = librosa.load(str(path), sr=sr, mono=True)
    return x.astype(np.float32), sr

def save_audio(dst, x, sr):
    dst.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(dst), x.astype(np.float32), sr)

def peak_normalize(x, eps=1e-9):
    m = np.max(np.abs(x)) + eps
    return (x / m).astype(np.float32)

# ---- Aumentos (sobre la onda) ----
def aug_time_stretch(x, sr, rate):
    y = librosa.effects.time_stretch(x, rate=rate)
    return peak_normalize(y), sr

def aug_pitch_shift(x, sr, n_steps):
    y = librosa.effects.pitch_shift(x, sr=sr, n_steps=n_steps)
    return peak_normalize(y), sr

def aug_gaussian_noise_snr(x, sr, snr_db):
    """
    Agrega ruido blanco con SNR objetivo (en dB):
      sigma = sqrt(Ps / 10^(SNR/10)), Ps = potencia de la señal
    """
    ps = float(np.mean(x**2) + 1e-12)
    sigma = np.sqrt(ps / (10**(snr_db/10)))
    n = rng.normal(0.0, sigma, size=x.shape).astype(np.float32)
    y = x + n
    return peak_normalize(y), sr


### Esquema de augmentations

In [ ]:
# Time-stretch ±15%  → 0.85 y 1.15
TIME_RATES = [0.85, 1.15]

# Pitch-shift ±2 a ±4 semitonos
PITCH_STEPS = [-4, -3, -2, 2, 3, 4]

# Ruido gaussiano con SNR en [15, 25] dB
NOISE_VARIANTS_PER_FILE = 5  # cuántas versiones con ruido generar por archivo
SNR_MIN, SNR_MAX = 15.0, 25.0


### Procesamiento (solo train/) y guardado

In [ ]:
files = [p for p in train_root.rglob('*') if p.suffix.lower() in EXTS]
if not files:
    raise RuntimeError(f'No se encontraron audios en {train_root}')

for src in tqdm(files, desc='Augmenting train/'):
    genre = src.relative_to(train_root).parts[0]
    stem  = src.stem

    # Cargar una sola vez
    x, sr = load_audio(src)

    # ---- Time-stretching ----
    for rate in TIME_RATES:
        try:
            y, sr_y = aug_time_stretch(x, sr, rate=rate)
            dst = out_root / genre / f'{stem}__ts_{int(rate*100):03d}.wav'  # p.ej. ts_085 / ts_115
            if WRITE: save_audio(dst, y, sr_y)
        except Exception as e:
            print(f'[TS {rate}] {src.name}: {e}')

    # ---- Pitch-shifting ----
    for steps in PITCH_STEPS:
        try:
            y, sr_y = aug_pitch_shift(x, sr, n_steps=steps)
            tag = f'ps_{steps:+d}'.replace('+','p').replace('-','m')  # ps_p3, ps_m4
            dst = out_root / genre / f'{stem}__{tag}.wav'
            if WRITE: save_audio(dst, y, sr_y)
        except Exception as e:
            print(f'[PS {steps}] {src.name}: {e}')

    # ---- Ruido gaussiano (SNR 15–25 dB) ----
    for k in range(NOISE_VARIANTS_PER_FILE):
        try:
            snr = float(rng.uniform(SNR_MIN, SNR_MAX))
            y, sr_y = aug_gaussian_noise_snr(x, sr, snr_db=snr)
            dst = out_root / genre / f'{stem}__noise_{snr:.1f}dB_{k+1}.wav'
            if WRITE: save_audio(dst, y, sr_y)
        except Exception as e:
            print(f'[NOISE] {src.name}: {e}')

print('Aumentos guardados en:', out_root)


### Unir originales + aumentados para entrenar

In [ ]:
import shutil

train_plus = train_root.parent / 'train_plus_aug'  # combinación final
for src in train_root.rglob('*'):
    if src.is_file() and src.suffix.lower() in EXTS:
        rel = src.relative_to(train_root)
        dst = train_plus / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

for src in out_root.rglob('*.wav'):
    rel = src.relative_to(out_root)
    dst = train_plus / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

print('Carpeta combinada para entrenamiento:', train_plus)


## Log-Mel2D

### Configuración

In [11]:
# %pip install librosa soundfile tqdm pillow --quiet

from pathlib import Path

SPLIT_DIR = Path(r'C:\Users\RICHI\Documents\Tesis\Data\Andinas\Segmentacion20seg_preenfasis_split')

# Qué carpetas procesar (en orden). Ajusta según tu caso:
input_folders = [
    SPLIT_DIR / 'train',  # si no existe, ignora
    SPLIT_DIR / 'val',
    SPLIT_DIR / 'test',
]

# Carpeta de salida (se espejo la jerarquía)
OUT_ROOT  = Path(r"C:\Users\RICHI\Documents\Tesis\Data\Andinas\Segmentacion20seg_preenfasis_Mel2D")  # <- salida

# Extensiones de audio
EXTS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a', '.aac'}
SPLITS    = ["train", "val", "test"]   # qué splits procesar
# Opciones de guardado
SAVE_NPY = False   # guarda matrices normalizadas (float32) .npy
SAVE_PNG = True   # guarda imagen 8-bit para inspección/uso visual
PNG_CLIP = 3.0    # recorte de z-score [-clip, +clip] antes de mapear a 0..255


### Funciones (STFT Hann → Mel 128 → dB → z-score por banda)

In [6]:
import numpy as np
import librosa, soundfile as sf
from tqdm.auto import tqdm
from PIL import Image

def pick_stft_params(sr: int):
    if sr <= 24000:
        n_fft = 1024
        hop_length = int(sr * 0.023)  # ~23 ms
    elif sr <= 44100:
        n_fft = 2048
        hop_length = int(sr * 0.012)  # ~12 ms (entre 10 y 23)
    else:
        n_fft = 4096
        hop_length = int(sr * 0.010)  # ~10 ms
    hop_length = max(1, hop_length)
    win_length = n_fft                  # ventana Hann de tamaño n_fft
    return n_fft, hop_length, win_length

def wav_to_mel_log2d(path, n_mels=128):
    # 1) Carga mono preservando sr original
    x, sr = librosa.load(str(path), sr=None, mono=True)
    # 2) STFT con ventana Hann
    n_fft, hop_length, win_length = pick_stft_params(sr)
    S = librosa.stft(
        x, n_fft=n_fft, hop_length=hop_length, win_length=win_length, window='hann', center=True
    )
    # 3) Espectrograma de potencia
    S_power = np.abs(S) ** 2
    # 4) Banco Mel (128 bandas) y proyección
    M = librosa.feature.melspectrogram(S=S_power, sr=sr, n_mels=n_mels, power=1.0)  # power ya aplicado
    # 5) Escala log (dB)
    M_db = librosa.power_to_db(M, ref=np.max)  # normaliza relativo al máximo
    # 6) Normalización por banda (eje frecuencia)
    #    z-score por cada banda mel a lo largo del tiempo
    mean = M_db.mean(axis=1, keepdims=True)
    std  = M_db.std(axis=1, keepdims=True) + 1e-8
    M_z = (M_db - mean) / std
    return M_z.astype(np.float32)  # [n_mels, T]



### Guardado

In [7]:
def save_mel(out_path_base: Path, M_z: np.ndarray, save_npy=True, save_png=True, clip=3.0):
    out_path_base.parent.mkdir(parents=True, exist_ok=True)
    if save_npy:
        np.save(str(out_path_base) + ".npy", M_z.astype(np.float32))
    if save_png:
        Mc  = np.clip(M_z, -clip, clip)
        M01 = (Mc - (-clip)) / (2 * clip)
        img = (M01 * 255.0).round().astype(np.uint8)
        Image.fromarray(img).save(str(out_path_base) + ".png")


### Procesamiento por lotes (preserva splits y géneros)

In [12]:
def process_split(split_root: Path, out_root: Path):
    """
    split_root = .../train  (o val/test)
    out_root   = .../SEGMENTACION..._mel_log2d
    Salida: out_root/<split>/<clase>/.../<archivo>.npy|.png
    """
    if not split_root.exists():
        print(f"[SKIP] No existe: {split_root}")
        return

    split_name = split_root.name  # 'train' | 'val' | 'test'
    files = [p for p in split_root.rglob("*") if p.suffix.lower() in EXTS]
    if not files:
        print(f"[AVISO] Sin archivos de audio en {split_root}")
        return

    ok = fail = 0
    for src in tqdm(files, desc=f"Mel-log2D: {split_name}"):
        # Relativo al split → <clase>/.../archivo.ext
        rel = src.relative_to(split_root)
        # Construye salida preservando split y clase:
        out_base = (out_root / split_name / rel).with_suffix("")  # quita extensión
        try:
            M = wav_to_mel_log2d(src, n_mels=128)   # tu función devuelve M_z
            save_mel(out_base, M, save_npy=SAVE_NPY, save_png=SAVE_PNG, clip=PNG_CLIP)
            ok += 1
        except Exception as e:
            print(f"[ERROR] {src} -> {e}")
            fail += 1
    print(f"[{split_name}] ok={ok} | errores={fail} | total={ok+fail}")

# --- Ejecuta para cada split ---
for split in SPLITS:
    process_split(SPLIT_DIR / split, OUT_ROOT)

print("Listo. Mel-log2D guardados en:", OUT_ROOT)


Mel-log2D: train:   0%|          | 0/29358 [00:00<?, ?it/s]

[train] ok=29358 | errores=0 | total=29358


Mel-log2D: val:   0%|          | 0/600 [00:00<?, ?it/s]

[val] ok=600 | errores=0 | total=600


Mel-log2D: test:   0%|          | 0/300 [00:00<?, ?it/s]

[test] ok=300 | errores=0 | total=300
Listo. Mel-log2D guardados en: C:\Users\RICHI\Documents\Tesis\Data\Andinas\Segmentacion20seg_preenfasis_Mel2D


# Modelo ViT

In [2]:
import platform, sys
print(platform.platform())
print("Python:", sys.version)
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Windows-10-10.0.26100-SP0
Python: 3.10.19 | packaged by conda-forge | (main, Oct 22 2025, 22:23:22) [MSC v.1944 64 bit (AMD64)]
Torch: 2.9.0+cu128
CUDA available: True


### 1) Conf

In [8]:
from pathlib import Path
import os, warnings, random, numpy as np, torch
warnings.filterwarnings("ignore")

# ====== AJUSTA ESTA RUTA ======
DATA_ROOT = Path(r"C:\Users\RICHI\Documents\Tesis\Data\Andinas\Segmentacion20seg_preenfasis_Mel2D")   # debe contener train/ val/ test/ (o train_plus_aug/ val/ test/)
CANDIDATOS_TRAIN = ["train_plus_aug", "train", "train_aug"]
VAL_NAME  = "val"
TEST_NAME = "test"

# Entrenamiento
BATCH_SIZE   = 32
NUM_WORKERS  = 0 if os.name == "nt" else 4   # en Windows, usa 0 si ves problemas
MAX_EPOCHS   = 80
LR           = 3e-4
WEIGHT_DECAY = 1e-4
DROP_RATE    = 0.10
SEED         = 42

# ViT
IMG_SIZE   = 224
PATCH_SIZE = 16
EMBED_DIM  = 384
DEPTH      = 8
NUM_HEADS  = 6
MLP_RATIO  = 4.0

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything()
torch.set_float32_matmul_precision("high")

# Verificación de estructura
if not DATA_ROOT.exists():
    raise FileNotFoundError(f"No existe DATA_ROOT: {DATA_ROOT}")
train_found = next((c for c in CANDIDATOS_TRAIN if (DATA_ROOT/c).exists()), None)
if not train_found:
    raise FileNotFoundError(f"No hallé carpeta de train en {DATA_ROOT}. Intenta con {CANDIDATOS_TRAIN}")
for split in [VAL_NAME, TEST_NAME]:
    if not (DATA_ROOT/split).exists():
        raise FileNotFoundError(f"Falta carpeta: {DATA_ROOT/split}")
SPLITS = {"train": train_found, "val": VAL_NAME, "test": TEST_NAME}
print("Usando SPLITS:", SPLITS)


Usando SPLITS: {'train': 'train', 'val': 'val', 'test': 'test'}


### 2) Dataset PNG (1 canal) + DataModule

In [9]:
import pytorch_lightning as pl
import torchvision
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class MelLog2DDataset(Dataset):
    def __init__(self, root: Path, size=224, class_to_idx=None):
        self.root = Path(root)
        if not self.root.exists():
            raise FileNotFoundError(f"No existe: {self.root}")
        self.size = size

        # Si NO me dan mapping, lo creo desde este root (se usará solo para TRAIN)
        if class_to_idx is None:
            classes = sorted([d.name for d in self.root.iterdir() if d.is_dir()])
            if not classes: raise RuntimeError(f"Sin subcarpetas en {self.root}")
            self.class_to_idx = {c:i for i,c in enumerate(classes)}
        else:
            self.class_to_idx = dict(class_to_idx)

        self.samples = []
        for c, idx in self.class_to_idx.items():
            cdir = self.root / c
            if not cdir.exists():  # clase que no aparece en este split
                continue
            for p in cdir.rglob("*.png"):
                self.samples.append((p, idx))
        if not self.samples:
            raise RuntimeError(f"Sin PNGs en {self.root} para las clases {list(self.class_to_idx.keys())}")

        self.resize = torchvision.transforms.Resize(
            (size, size), interpolation=torchvision.transforms.InterpolationMode.BILINEAR
        )

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, y = self.samples[idx]
        img = Image.open(path).convert('L')                        # 1 canal
        x = torchvision.transforms.functional.pil_to_tensor(img).float()/255.0  # [0,1]
        x = self.resize(x)                                         # (1,224,224)
        # 🔸 Normalización simple a [-1, 1]
        x = (x - 0.5) / 0.5
        return x, y

class MelLog2DDataModule(pl.LightningDataModule):
    def __init__(self, data_root: Path, splits: dict, batch_size=32, num_workers=0, size=224):
        super().__init__()
        self.data_root, self.splits = Path(data_root), splits
        self.bs, self.nw, self.size = batch_size, num_workers, size
        self.class_to_idx = None
        self.num_classes = None

    def setup(self, stage=None):
        # 1) Construye mapping SOLO desde train
        train_root = self.data_root / self.splits['train']
        classes = sorted([d.name for d in train_root.iterdir() if d.is_dir()])
        if not classes: raise RuntimeError(f"Sin clases en {train_root}")
        self.class_to_idx = {c:i for i,c in enumerate(classes)}
        self.num_classes  = len(self.class_to_idx)

        # 2) Reutiliza el mapping en todos los splits
        self.ds_train = MelLog2DDataset(train_root, self.size, class_to_idx=self.class_to_idx)
        self.ds_val   = MelLog2DDataset(self.data_root/self.splits['val'],  self.size, class_to_idx=self.class_to_idx)
        self.ds_test  = MelLog2DDataset(self.data_root/self.splits['test'], self.size, class_to_idx=self.class_to_idx)

        print("Clases (orden fijo):", classes)
        print(f"train={len(self.ds_train)} | val={len(self.ds_val)} | test={len(self.ds_test)}")

    def train_dataloader(self):
        return DataLoader(self.ds_train, batch_size=self.bs, shuffle=True,  num_workers=self.nw, pin_memory=True)
    def val_dataloader(self):
        return DataLoader(self.ds_val,   batch_size=self.bs, shuffle=False, num_workers=self.nw, pin_memory=True)
    def test_dataloader(self):
        return DataLoader(self.ds_test,  batch_size=self.bs, shuffle=False, num_workers=self.nw, pin_memory=True)

### 3) ViT minimal (parches 16×16, [CLS], pos emb, L bloques)

In [10]:
import torch.nn as nn
import torch

class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=1, embed_dim=384):
        super().__init__()
        assert img_size % patch_size == 0, "img_size debe ser múltiplo de patch_size"
        self.grid = img_size // patch_size
        self.num_patches = self.grid * self.grid
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):             # (B,1,224,224)
        x = self.proj(x)              # (B,D,14,14)
        x = x.flatten(2).transpose(1,2)  # (B,N,D) con N=196
        return x

class EncoderBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, drop=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, num_heads, dropout=drop, batch_first=True)
        self.drop1 = nn.Dropout(drop)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden, dim), nn.Dropout(drop),
        )

    def forward(self, x):             # (B,1+N,D)
        h = self.norm1(x)
        a, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.drop1(a)         # residual
        h = self.norm2(x)
        x = x + self.mlp(h)           # residual
        return x

class ViT(nn.Module):
    def __init__(self, num_classes, img_size=224, patch_size=16, in_chans=1,
                 embed_dim=384, depth=8, num_heads=6, mlp_ratio=4.0, drop=0.1):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        N = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1,1,embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, N+1, embed_dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.pos_drop = nn.Dropout(drop)
        self.blocks = nn.ModuleList([EncoderBlock(embed_dim, num_heads, mlp_ratio, drop) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):             # (B,1,224,224)
        B = x.size(0)
        x = self.patch_embed(x)       # (B,N,D)
        cls = self.cls_token.expand(B, -1, -1)      # (B,1,D)
        x = torch.cat([cls, x], dim=1)              # (B,1+N,D)
        x = x + self.pos_embed[:, :x.size(1), :]
        x = self.pos_drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        return self.head(x[:,0])      # logits del [CLS]

### 4) LightningModule (CE + AdamW) y CSVLogger

In [11]:
from torchmetrics.classification import MulticlassAccuracy
from pytorch_lightning.loggers import CSVLogger
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from torchmetrics.classification import (
    MulticlassAccuracy, MulticlassPrecision, MulticlassRecall, MulticlassF1Score
)

class ViTClassifierPL(pl.LightningModule):
    def __init__(self, num_classes, lr=3e-4, wd=1e-4,
                 img_size=224, patch_size=16, embed_dim=384, depth=8, num_heads=6, mlp_ratio=4.0, drop=0.1):
        super().__init__()
        self.save_hyperparameters()
        self.model = ViT(num_classes, img_size, patch_size, 1, embed_dim, depth, num_heads, mlp_ratio, drop)
        self.crit  = nn.CrossEntropyLoss()

        self.train_acc  = MulticlassAccuracy(num_classes=num_classes)
        self.val_acc    = MulticlassAccuracy(num_classes=num_classes)
        self.test_acc   = MulticlassAccuracy(num_classes=num_classes)

        self.train_prec = MulticlassPrecision(num_classes=num_classes, average="macro")
        self.val_prec   = MulticlassPrecision(num_classes=num_classes, average="macro")
        self.test_prec  = MulticlassPrecision(num_classes=num_classes, average="macro")

        self.train_rec  = MulticlassRecall(num_classes=num_classes, average="macro")
        self.val_rec    = MulticlassRecall(num_classes=num_classes, average="macro")
        self.test_rec   = MulticlassRecall(num_classes=num_classes, average="macro")

        self.train_f1   = MulticlassF1Score(num_classes=num_classes, average="macro")
        self.val_f1     = MulticlassF1Score(num_classes=num_classes, average="macro")
        self.test_f1    = MulticlassF1Score(num_classes=num_classes, average="macro")

    def forward(self, x): return self.model(x)

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.wd)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.trainer.max_epochs)
        return {"optimizer": opt, "lr_scheduler": sch}

    def _step(self, batch, stage: str):
        x, y = batch
        logits = self(x)
        loss = self.crit(logits, y)
        preds = logits.argmax(dim=1)

        acc  = getattr(self, f"{stage}_acc")(preds, y)
        prec = getattr(self, f"{stage}_prec")(preds, y)
        rec  = getattr(self, f"{stage}_rec")(preds, y)
        f1   = getattr(self, f"{stage}_f1")(preds, y)

        self.log(f"{stage}/loss", loss, prog_bar=(stage!='test'), on_epoch=True, on_step=False)
        self.log(f"{stage}/acc",  acc,  prog_bar=(stage!='test'), on_epoch=True, on_step=False)
        self.log(f"{stage}/prec", prec, prog_bar=False,          on_epoch=True, on_step=False)
        self.log(f"{stage}/rec",  rec,  prog_bar=False,          on_epoch=True, on_step=False)
        self.log(f"{stage}/f1",   f1,   prog_bar=True,           on_epoch=True, on_step=False)

        return loss

    def training_step(self, b, _): return self._step(b, "train")
    def validation_step(self, b, _): return self._step(b, "val")
    def test_step(self, b, _): return self._step(b, "test")

In [12]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i} ->", torch.cuda.get_device_name(i))
print("Current device index:", torch.cuda.current_device())


CUDA available: True
CUDA device count: 1
GPU 0 -> NVIDIA GeForce RTX 3050 Laptop GPU
Current device index: 0


### 5) Entrenar / Validar / Testear con CSVLogger

#### Resumen de época por consola

In [13]:
class EpochSummaryCallback(pl.Callback):
    def on_validation_epoch_end(self, trainer, pl_module):
        m = trainer.callback_metrics
        def g(k): 
            return float(m[k]) if k in m and m[k] is not None else None
        ep = trainer.current_epoch
        keys = ["train/loss","train/acc","train/prec","train/rec","train/f1",
                "val/loss","val/acc","val/prec","val/rec","val/f1"]
        msg = f"[Epoch {ep:03d}] " + " | ".join([f"{k}={g(k):.4f}" for k in keys if g(k) is not None])
        print(msg)


#### Verificación de estructura

In [14]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(f"No existe DATA_ROOT: {DATA_ROOT}")
train_found = next((c for c in CANDIDATOS_TRAIN if (DATA_ROOT/c).exists()), None)
if not train_found:
    raise FileNotFoundError(f"No hallé carpeta de train en {DATA_ROOT}. Intenta con {CANDIDATOS_TRAIN}")
for split in [VAL_NAME, TEST_NAME]:
    if not (DATA_ROOT/split).exists():
        raise FileNotFoundError(f"Falta carpeta: {DATA_ROOT/split}")
SPLITS = {"train": train_found, "val": VAL_NAME, "test": TEST_NAME}
print("Usando SPLITS:", SPLITS)

Usando SPLITS: {'train': 'train', 'val': 'val', 'test': 'test'}


#### Entrenamiento / Test

In [15]:
def run_training():
    dm = MelLog2DDataModule(DATA_ROOT, SPLITS, BATCH_SIZE, NUM_WORKERS, IMG_SIZE)
    dm.setup()
    num_classes = dm.num_classes

    model = ViTClassifierPL(
        num_classes=num_classes, lr=LR, wd=WEIGHT_DECAY,
        img_size=IMG_SIZE, patch_size=PATCH_SIZE,
        embed_dim=EMBED_DIM, depth=DEPTH, num_heads=NUM_HEADS,
        mlp_ratio=MLP_RATIO, drop=DROP_RATE
    )

    ckpt  = ModelCheckpoint(monitor='val/acc', mode='max',
                            save_top_k=1, save_last=True,
                            filename='vit-{epoch:02d}-{val_acc:.3f}')
    early = EarlyStopping(monitor='val/acc', mode='max', patience=10)
    lrmon = LearningRateMonitor(logging_interval='epoch')
    logger = CSVLogger("logs", name="vit")
    epoch_summary = EpochSummaryCallback()

    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=[0] if torch.cuda.is_available() else 'auto',
        precision='16-mixed' if torch.cuda.is_available() else 32,
        callbacks=[ckpt, early, lrmon, epoch_summary],
        logger=[logger],
        log_every_n_steps=10
    )

    print("Batches/epoch -> train:", len(dm.train_dataloader()),
          "| val:", len(dm.val_dataloader()), "| test:", len(dm.test_dataloader()))

    trainer.fit(model, datamodule=dm)

    best_path = ckpt.best_model_path
    print("Best ckpt:", best_path or "<none>", "| best score:", ckpt.best_model_score)

    # Usa 'best' si existe; si no, pesos actuales
    trainer.test(model=None, datamodule=dm, ckpt_path=('best' if best_path else None))
    print("Logs CSV en:", logger.log_dir)  # metrics.csv con loss/acc/prec/rec/f1 por época

if __name__ == "__main__":
    run_training()

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Clases (orden fijo): ['Chimayche', 'Cumbia andina', 'Diablada', 'Huaylash', 'Huayno', 'Muliza', 'Santiago', 'Tinku', 'Tunantada', 'Yaravi (Harawi)']
train=29358 | val=600 | test=300
Batches/epoch -> train: 918 | val: 19 | test: 10


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name       | Type                | Params | Mode 
------------------------------------------------------------
0  | model      | ViT                 | 14.4 M | train
1  | crit       | CrossEntropyLoss    | 0      | train
2  | train_acc  | MulticlassAccuracy  | 0      | train
3  | val_acc    | MulticlassAccuracy  | 0      | train
4  | test_acc   | MulticlassAccuracy  | 0      | train
5  | train_prec | MulticlassPrecision | 0      | train
6  | val_prec   | MulticlassPrecision | 0      | train
7  | test_prec  | MulticlassPrecision | 0      | train
8  | train_rec  | MulticlassRecall    | 0      | train
9  | val_rec    | MulticlassRecall    | 0      | train
10 | test_rec   | MulticlassRecall    | 0      | train
11 | train_f1   | MulticlassF1Score   | 0      | train
12 | val_f1     | MulticlassF1Score   | 0      | train
13 | test_f1    | MulticlassF1Score   | 0      | train
------------------------------------------------------------
14.4 M    

Clases (orden fijo): ['Chimayche', 'Cumbia andina', 'Diablada', 'Huaylash', 'Huayno', 'Muliza', 'Santiago', 'Tinku', 'Tunantada', 'Yaravi (Harawi)']
train=29358 | val=600 | test=300


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

[Epoch 029] train/loss=0.0512 | train/acc=0.9753 | train/prec=0.9759 | train/rec=0.9753 | train/f1=0.9721 | val/loss=2.4775 | val/acc=0.1519 | val/prec=0.2341 | val/rec=0.1519 | val/f1=0.1798
Best ckpt: logs\vit\version_0\checkpoints\vit-epoch=19-val_acc=0.000.ckpt | best score: tensor(0.1708, device='cuda:0')


Restoring states from the checkpoint path at logs\vit\version_0\checkpoints\vit-epoch=19-val_acc=0.000.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at logs\vit\version_0\checkpoints\vit-epoch=19-val_acc=0.000.ckpt


Clases (orden fijo): ['Chimayche', 'Cumbia andina', 'Diablada', 'Huaylash', 'Huayno', 'Muliza', 'Santiago', 'Tinku', 'Tunantada', 'Yaravi (Harawi)']
train=29358 | val=600 | test=300


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test/acc            0.24480150640010834
         test/f1            0.2701926529407501
        test/loss            1.493787169456482
        test/prec           0.31015440821647644
        test/rec            0.24480150640010834
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Logs CSV en: logs\vit\version_0
